In [1]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family']  = 'DejaVu Sans'


In [2]:
adata_a = sc.read('../adatas/iAstrocytes_SCREEN1_filtered.h5ad')
print (adata_a.X.min(),adata_a.X.max())
adata_a

0.0 7344.0


AnnData object with n_obs × n_vars = 195306 × 36601
    obs: 'target_gene', 'guide_vector', 'sequencing_batch'

In [2]:
adata_n = sc.read('../preprocessing/adatas/iNeurons_SCREEN1_filtered_INT3INT30removed.h5ad')
print (adata_n.X.min(), adata_n.X.max())
adata_n

0.0 6620.0


AnnData object with n_obs × n_vars = 204719 × 36601
    obs: 'Sample', 'Replicate', 'num_features', 'feature_call', 'num_umis', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'guide_calls', 'target_vector', 'target_gene', 'perturbation'
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'target_gene_id', 'target_gene_name', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'Replicate_colors'

In [3]:
sc.pp.normalize_total(adata_a)
sc.pp.log1p(adata_a)
sc.pp.highly_variable_genes(adata_a, subset=True)

In [3]:
sc.pp.normalize_total(adata_n)
sc.pp.log1p(adata_n)
sc.pp.highly_variable_genes(adata_n, subset=True)

In [8]:
adata_n.X.shape

(204719, 1532)

In [4]:
genes = list(adata_a.obs.target_gene.unique())

In [5]:
genes = [g for g in genes if g != 'INT']

In [6]:
import re
from scipy.stats import pearsonr

os.makedirs("figures/correlation_iA", exist_ok=True)

summary = []   # <-- add this

genes = list(adata_a.obs.target_gene.unique())
perts = sorted([g for g in genes if g != 'INT'])

for pert in perts:
    temp = adata_a[adata_a.obs["target_gene"] == pert]

    raw_groups = pd.Index(temp.obs["guide_vector"].dropna().unique())

    # prescriptive order for this perturbation
    expected = [f"{pert}_A", f"{pert}_B", f"{pert}_C"]

    # keep only the guides that actually exist, in the expected order
    groups = [g for g in expected if g in set(raw_groups)]
    if len(groups) < 2:
        print(f"{pert}: <2 guides (after ordering), skipping")
        continue

    # counts per guide (x/y axis labels will include these)
    counts = temp.obs["guide_vector"].value_counts()

    # gene × guide means
    genes = temp.var_names
    result = pd.DataFrame(index=genes, columns=groups, dtype=float)
    for g in groups:
        mask = temp.obs["guide_vector"] == g
        X_sub = temp[mask, genes].X
        result[g] = np.asarray(X_sub.mean(axis=0)).ravel()
    R = result.apply(pd.to_numeric, errors="coerce")

    if len(groups) == 2:
        fig, ax = plt.subplots(figsize=(4, 4))
        xcol, ycol = groups[0], groups[1]
        xy = R[[xcol, ycol]].dropna()
        if xy.empty:
            plt.close(fig)
            continue

        x, y = xy[xcol].to_numpy(float), xy[ycol].to_numpy(float)
        ax.scatter(x, y, s=5)
        ax.plot([0, 3], [0, 3], "k--", lw=0.8)
        ax.set_xlim(0, 3)
        ax.set_ylim(0, 3)

        if x.size >= 2 and np.std(x) > 0 and np.std(y) > 0:
            r, p = pearsonr(x, y)
            ax.text(
                0.05, 0.95, f"r={r:.2f}",
                transform=ax.transAxes,
                ha="left", va="top", fontsize=8
            )
        else:
            r, p = np.nan, np.nan

        summary.append({
            "Perturbation": pert,
            "Guide_1": xcol,
            "Guide_2": ycol,
            "Guide_1_cells": counts.get(xcol, 0),
            "Guide_2_cells": counts.get(ycol, 0),
            "Pearson_r": r,
            "Pearson_p": p,
            "Genes_used": len(xy),
        })

        ax.set_xlabel(f"{xcol} (n={counts.get(xcol, 0)})", fontsize=8)
        ax.set_ylabel(f"{ycol} (n={counts.get(ycol, 0)})", fontsize=8)

        fig.tight_layout()
        fig.savefig(f"figures/correlation_iA/2_{pert}.pdf")
        plt.close(fig)

    else:
        # use first three guides for the triangle
        cols = list(groups)[:3]
        R = R[cols]

        fig, axes = plt.subplots(2, 2, figsize=(4, 4))
        axes[0, 1].axis("off")  # top-right blank

        mapping = {(0, 1): axes[0, 0], (0, 2): axes[1, 0], (1, 2): axes[1, 1]}

        for (i, j), ax in mapping.items():
            xcol, ycol = cols[i], cols[j]
            xy = R[[xcol, ycol]].dropna()

            if xy.empty:
                ax.set_axis_off()
                continue

            x, y = xy[xcol].to_numpy(float), xy[ycol].to_numpy(float)
            ax.scatter(x, y, s=5)
            ax.plot([0, 3], [0, 3], "k--", lw=0.8)
            ax.set_xlim(0, 3)
            ax.set_ylim(0, 3)

            if x.size >= 2 and np.std(x) > 0 and np.std(y) > 0:
                r, p = pearsonr(x, y)
                ax.text(
                    0.05, 0.95, f"r={r:.2f}",
                    transform=ax.transAxes,
                    ha="left", va="top", fontsize=8
                )
            else:
                r, p = np.nan, np.nan

            summary.append({
                "Perturbation": pert,
                "Guide_1": xcol,
                "Guide_2": ycol,
                "Guide_1_cells": counts.get(xcol, 0),
                "Guide_2_cells": counts.get(ycol, 0),
                "Pearson_r": r,
                "Pearson_p": p,
                "Genes_used": len(xy),
            })

            ax.set_xlabel(f"{xcol} (n={counts.get(xcol, 0)})", fontsize=8)
            ax.set_ylabel(f"{ycol} (n={counts.get(ycol, 0)})", fontsize=8)

        fig.tight_layout()
        fig.savefig(f"figures/correlation_iA/2_{pert}.pdf")
        plt.close(fig)

# Save supplementary table
summary_df = pd.DataFrame(summary)
summary_df.to_csv(
    "figures/correlation_iA/guide_correlations.csv",
    index=False,
)

In [10]:
adata_n.obs

,Sample,Replicate,num_features,feature_call,num_umis,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,total_counts_ribo,pct_counts_ribo,total_counts_hb,pct_counts_hb,guide_calls,target_vector,target_gene,perturbation
cellranger801_count_6cb53aeb358675da37acacb559bfa791___AAACCTGAGAAGAAGC-1,cellranger801_count_6cb53aeb358675da37acacb559...,Screen1_0,1.0,UNC13A_A_1,115,3618,6979.0,138.0,1.977361,283.0,4.055022,0.0,0.000000,Single_guide,UNC13A_A,UNC13A,UNC13A
cellranger801_count_6cb53aeb358675da37acacb559bfa791___AAACCTGAGACAAGCC-1,cellranger801_count_6cb53aeb358675da37acacb559...,Screen1_0,1.0,QPCT_B_1,375,3650,7800.0,242.0,3.102564,431.0,5.525641,0.0,0.000000,Single_guide,QPCT_B,QPCT,QPCT
cellranger801_count_6cb53aeb358675da37acacb559bfa791___AAACCTGAGACTGTAA-1,cellranger801_count_6cb53aeb358675da37acacb559...,Screen1_0,1.0,HLA-DRA_C_1,7,4505,10905.0,291.0,2.668501,562.0,5.153599,0.0,0.000000,Single_guide,HLA-DRA_C,HLA-DRA,HLA-DRA
cellranger801_count_6cb53aeb358675da37acacb559bfa791___AAACCTGAGATATGGT-1,cellranger801_count_6cb53aeb358675da37acacb559...,Screen1_0,2.0,RASGRF1_B_1|RASGRF1_B_2,25|26,5351,14125.0,411.0,2.909734,557.0,3.943363,0.0,0.000000,Correct_dual_guides,RASGRF1_B,RASGRF1,RASGRF1
cellranger801_count_6cb53aeb358675da37acacb559bfa791___AAACCTGAGGTCGGAT-1,cellranger801_count_6cb53aeb358675da37acacb559...,Screen1_0,1.0,C1orf159_B_1,36,4604,10758.0,119.0,1.106153,498.0,4.629113,1.0,0.009295,Single_guide,C1orf159_B,C1orf159,C1orf159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49352_NID_scRNA15027413_and_49352_NID_scRNA15027416___TTTGTCATCAGCGACC-1,49352_NID_scRNA15027413_and_49352_NID_scRNA150...,Screen1_4,1.0,BIN1_C_1,3,2009,3085.0,115.0,3.727715,101.0,3.273906,0.0,0.000000,Single_guide,BIN1_C,BIN1,BIN1
49352_NID_scRNA15027413_and_49352_NID_scRNA15027416___TTTGTCATCCAGAAGG-1,49352_NID_scRNA15027413_and_49352_NID_scRNA150...,Screen1_4,1.0,VPS35_B_1,5,2984,5245.0,118.0,2.249762,216.0,4.118208,0.0,0.000000,Single_guide,VPS35_B,VPS35,VPS35
49352_NID_scRNA15027413_and_49352_NID_scRNA15027416___TTTGTCATCCTATGTT-1,49352_NID_scRNA15027413_and_49352_NID_scRNA150...,Screen1_4,1.0,AARD_A_1,115,3684,6978.0,180.0,2.579536,193.0,2.765836,2.0,0.028662,Single_guide,AARD_A,AARD,AARD
49352_NID_scRNA15027413_and_49352_NID_scRNA15027416___TTTGTCATCGGCTACG-1,49352_NID_scRNA15027413_and_49352_NID_scRNA150...,Screen1_4,1.0,GRB10_A_2,3,2004,2978.0,75.0,2.518469,69.0,2.316991,0.0,0.000000,Single_guide,GRB10_A,GRB10,GRB10


In [9]:
import re
from scipy.stats import pearsonr

os.makedirs("figures/correlation_iN", exist_ok=True)

summary = []   # <-- add this

genes = list(adata_n.obs.perturbation.unique())
perts = sorted([g for g in genes if g != 'INT'])

for pert in perts:
    temp = adata_n[adata_n.obs["perturbation"] == pert]

    raw_groups = pd.Index(temp.obs["target_vector"].dropna().unique())

    # prescriptive order for this perturbation
    expected = [f"{pert}_A", f"{pert}_B", f"{pert}_C"]

    # keep only the guides that actually exist, in the expected order
    groups = [g for g in expected if g in set(raw_groups)]
    if len(groups) < 2:
        print(f"{pert}: <2 guides (after ordering), skipping")
        continue

    # counts per guide (x/y axis labels will include these)
    counts = temp.obs["target_vector"].value_counts()

    # gene × guide means
    genes = temp.var_names
    result = pd.DataFrame(index=genes, columns=groups, dtype=float)
    for g in groups:
        mask = temp.obs["target_vector"] == g
        X_sub = temp[mask, genes].X
        result[g] = np.asarray(X_sub.mean(axis=0)).ravel()
    R = result.apply(pd.to_numeric, errors="coerce")

    if len(groups) == 2:
        fig, ax = plt.subplots(figsize=(4, 4))
        xcol, ycol = groups[0], groups[1]
        xy = R[[xcol, ycol]].dropna()
        if xy.empty:
            plt.close(fig)
            continue

        x, y = xy[xcol].to_numpy(float), xy[ycol].to_numpy(float)
        ax.scatter(x, y, s=5)
        ax.plot([0, 3], [0, 3], "k--", lw=0.8)
        ax.set_xlim(0, 3)
        ax.set_ylim(0, 3)

        if x.size >= 2 and np.std(x) > 0 and np.std(y) > 0:
            r, p = pearsonr(x, y)
            ax.text(
                0.05, 0.95, f"r={r:.2f}",
                transform=ax.transAxes,
                ha="left", va="top", fontsize=8
            )
        else:
            r, p = np.nan, np.nan

        summary.append({
            "Perturbation": pert,
            "Guide_1": xcol,
            "Guide_2": ycol,
            "Guide_1_cells": counts.get(xcol, 0),
            "Guide_2_cells": counts.get(ycol, 0),
            "Pearson_r": r,
            "Pearson_p": p,
            "Genes_used": len(xy),
        })

        ax.set_xlabel(f"{xcol} (n={counts.get(xcol, 0)})", fontsize=8)
        ax.set_ylabel(f"{ycol} (n={counts.get(ycol, 0)})", fontsize=8)

        fig.tight_layout()
        fig.savefig(f"figures/correlation_iN/{pert}.pdf")
        plt.close(fig)

    else:
        # use first three guides for the triangle
        cols = list(groups)[:3]
        R = R[cols]

        fig, axes = plt.subplots(2, 2, figsize=(4, 4))
        axes[0, 1].axis("off")  # top-right blank

        mapping = {(0, 1): axes[0, 0], (0, 2): axes[1, 0], (1, 2): axes[1, 1]}

        for (i, j), ax in mapping.items():
            xcol, ycol = cols[i], cols[j]
            xy = R[[xcol, ycol]].dropna()

            if xy.empty:
                ax.set_axis_off()
                continue

            x, y = xy[xcol].to_numpy(float), xy[ycol].to_numpy(float)
            ax.scatter(x, y, s=5)
            ax.plot([0, 3], [0, 3], "k--", lw=0.8)
            ax.set_xlim(0, 3)
            ax.set_ylim(0, 3)

            if x.size >= 2 and np.std(x) > 0 and np.std(y) > 0:
                r, p = pearsonr(x, y)
                ax.text(
                    0.05, 0.95, f"r={r:.2f}",
                    transform=ax.transAxes,
                    ha="left", va="top", fontsize=8
                )
            else:
                r, p = np.nan, np.nan

            summary.append({
                "Perturbation": pert,
                "Guide_1": xcol,
                "Guide_2": ycol,
                "Guide_1_cells": counts.get(xcol, 0),
                "Guide_2_cells": counts.get(ycol, 0),
                "Pearson_r": r,
                "Pearson_p": p,
                "Genes_used": len(xy),
            })

            ax.set_xlabel(f"{xcol} (n={counts.get(xcol, 0)})", fontsize=8)
            ax.set_ylabel(f"{ycol} (n={counts.get(ycol, 0)})", fontsize=8)

        fig.tight_layout()
        fig.savefig(f"figures/correlation_iN/{pert}.pdf")
        plt.close(fig)

# Save supplementary table
summary_df = pd.DataFrame(summary)
summary_df.to_csv(
    "figures/correlation_iN/guide_correlations_iN.csv",
    index=False,
)